# 🧠 eidcloud-memory: Quickstart Guide

Welcome to **eidcloud-memory**, a multi-tiered AI agent memory engine with importance scoring, working memory, and episodic retrieval in pure PHP 8.2+.

This 3-cell notebook demonstrates:
1. **Environment Setup & Verification**
2. **Dialogue Turn Ingestion & Working Memory**
3. **Decay-Driven Memory Pruning & Associative Retrieval**

In [ ]:
# Cell 1: Environment Setup & Pure PHP Verification
!apt-get update -qq && apt-get install -y -qq php-cli php-sqlite3 > /dev/null
!php -v

# Clone or verify repository
!if [ ! -d "eidcloud-memory" ]; then git clone https://github.com/shadialhasan/eidcloud-memory.git; fi
%cd eidcloud-memory
!php tests/run_tests.php

In [ ]:
# Cell 2: Dialogue Ingestion & Multi-Tier Memory Storage
script = '''<?php
require_once 'src/autoload.php';

use EidCloud\\Memory\\MemoryManager;
use EidCloud\\Memory\\Models\\MemoryTier;
use EidCloud\\Memory\\Storage\\SqliteStorage;

$manager = new MemoryManager(new SqliteStorage('colab_demo.db'));

// 1. Ingest episodic dialogue turns
$manager->logEpisodicTurn('user', 'I am building an autonomous agent in Damascus.', 1, importance: 6.0);
$manager->logEpisodicTurn('assistant', 'I can help you manage state and multi-tier memory.', 2, importance: 5.0);
$manager->logEpisodicTurn('user', 'Please prioritize privacy and SQLite persistence.', 3, importance: 9.0, extraTags: ['pref', 'security']);

// 2. Set scratchpad working memory
$manager->setWorkingMemory('active_task', 'Synthesizing memory architecture', 8.5);

echo "Ingested dialogue turns & working memory successfully.\\n";
print_r($manager->getStats());
'''
with open('demo_step1.php', 'w') as f:
    f.write(script)

!php demo_step1.php

In [ ]:
# Cell 3: Recency/Importance Scoring, Pruning & Associative Retrieval
script = '''<?php
require_once 'src/autoload.php';

use EidCloud\\Memory\\MemoryManager;
use EidCloud\\Memory\\Storage\\SqliteStorage;

$manager = new MemoryManager(new SqliteStorage('colab_demo.db'));

// 1. Run Maintenance / Pruning
$report = $manager->pruneAndConsolidate();
echo "Maintenance Pruning Report: ", json_encode($report), "\\n\\n";

// 2. Ranked Retrieval with decay & relevance
$query = 'What are the user preferences on security and storage?';
echo "Query: '$query'\\n";
$results = $manager->recall($query, topK: 3);

foreach ($results as $item) {
    echo sprintf("[%s] [Score: %.3f] [Importance: %.1f] %s\\n", $item->tier->value, $item->score, $item->importance, $item->content);
}

// 3. Format Context for LLM prompt injection
echo "\\n--- Formatted Context Window for LLM ---\\n";
echo $manager->formatContextWindow($query);
'''
with open('demo_step2.php', 'w') as f:
    f.write(script)

!php demo_step2.php